# Task 04: Persistence and Historical-Average Baselines

This notebook is an educational, manually executed view of the tested Task 04 pipeline. The reusable implementation remains in `src/models/baselines.py`, shared metrics remain in `src/evaluation/metrics.py`, and artifact generation remains in `scripts/run_baselines.py`.

The experiment evaluates METR-LA test windows at **+15, +30, and +60 minutes** (forecast steps 3, 6, and 12). All errors are reported in original speed units (mph).

## 1. Scientific policies

- **Persistence:** repeat the final observed input speed at every future step.
- **Historical Average:** use a per-sensor mean for each five-minute slot of the week, fitted only from chronological training rows. An unseen slot uses that sensor's training-only finite-value mean.
- **Zero readings:** recorded zeros are preserved. The primary mask excludes exact-zero targets because Task 02 found their meaning uncertain; a sensitivity mask includes every finite target, including zero.
- **Fairness:** both baselines use identical test timestamps, sensors, horizons, targets, and masks. Validation/test observations never enter Historical Average fitting.

## 2. Set up the repository imports

Run this notebook from VS Code with the repository `.venv` kernel. The path search supports either the repository root or the `notebooks` directory as the kernel working directory.

In [1]:
import json
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / 'scripts' / 'run_baselines.py').exists():
    ROOT = ROOT.parent
if not (ROOT / 'scripts' / 'run_baselines.py').exists():
    raise FileNotFoundError('Run this notebook from the repository root or notebooks directory')

sys.path.insert(0, str(ROOT))
from scripts.run_baselines import main as run_task04_baselines

print('Repository root resolved successfully.')

Repository root resolved successfully.


## 3. Generate the reproducible Task 04 artifacts

This calls the same script used by the command-line workflow. It reads the verified CSV, recreates the fixed chronological splits and 12-to-12 windows, fits Historical Average on training rows only, evaluates both mask policies, and writes the metrics and compressed predictions. No random operations or model training are involved.

In [2]:
run_task04_baselines()

Saved artifacts\task04_baselines\test_predictions_horizons.npz (11436251 bytes)
Saved results\task04_baselines\metrics.json
persistence +15: MAE=4.017540 mph, RMSE=8.695499 mph, n=1245040
persistence +30: MAE=5.095234 mph, RMSE=11.135297 mph, n=1245022
persistence +60: MAE=6.798028 mph, RMSE=14.213606 mph, n=1244986
historical_average +15: MAE=7.110352 mph, RMSE=9.831092 mph, n=1245040
historical_average +30: MAE=7.110392 mph, RMSE=9.831118 mph, n=1245022
historical_average +60: MAE=7.110807 mph, RMSE=9.831346 mph, n=1244986


## 4. Load and verify the saved outputs

The NPZ contains the targets, forecasts, timestamps, sensor IDs, horizons, and both masks. The JSON contains human-readable metrics and provenance, including split boundaries and SHA-256 hashes.

In [3]:
metrics_path = ROOT / 'results' / 'task04_baselines' / 'metrics.json'
predictions_path = ROOT / 'artifacts' / 'task04_baselines' / 'test_predictions_horizons.npz'

metrics_document = json.loads(metrics_path.read_text(encoding='utf-8'))
predictions = np.load(predictions_path, allow_pickle=False)

expected_windows = metrics_document['splits']['test']['row_count'] - 11
assert predictions['targets_mph'].shape == (expected_windows, 3, 207)
assert predictions['persistence_mph'].shape == predictions['targets_mph'].shape
assert predictions['historical_average_mph'].shape == predictions['targets_mph'].shape
assert np.array_equal(predictions['horizon_minutes'], [15, 30, 60])

print(f"Prediction tensor: {predictions['targets_mph'].shape} (windows, horizons, sensors)")
print(f"Sensors: {len(predictions['sensor_ids'])}")
print(f"Forecast origins: {predictions['forecast_origin_timestamps'][0]} to {predictions['forecast_origin_timestamps'][-1]}")
print(f"Prediction SHA-256: {metrics_document['prediction_sha256']}")

Prediction tensor: (6844, 3, 207) (windows, horizons, sensors)
Sensors: 207
Forecast origins: 2012-06-04T04:40:00.000000 to 2012-06-27T22:55:00.000000
Prediction SHA-256: 48FD54D29C5BDB130F978F2526865649BF247CE8D6C371C8807EE8DC4D5B4321


## 5. Compare primary masked MAE and RMSE

The primary policy excludes only exact-zero and non-finite targets. It does not remove difficult nonzero traffic conditions, and it is identical for both models. Lower values are better.

In [4]:
metrics = pd.DataFrame(metrics_document['metrics'])
primary = metrics.loc[
    metrics['mask_policy'].eq('finite_nonzero_targets'),
    ['model', 'horizon', 'forecast_step_1_based', 'eligible_count', 'mae_mph', 'rmse_mph'],
].copy()
primary

,model,horizon,forecast_step_1_based,eligible_count,mae_mph,rmse_mph
0,persistence,+15,3,1245040,4.017540,8.695499
1,persistence,+30,6,1245022,5.095234,11.135297
2,persistence,+60,12,1244986,6.798028,14.213606
6,historical_average,+15,3,1245040,7.110352,9.831092
7,historical_average,+30,6,1245022,7.110392,9.831118
8,historical_average,+60,12,1244986,7.110807,9.831346


## 6. Inspect zero-target sensitivity

This table places the primary and zero-inclusive results side by side. A material change shows why the zero policy must be reported explicitly rather than silently treating all zeros as missing or all zeros as genuine stopped traffic.

In [5]:
comparison = metrics.pivot(
    index=['model', 'horizon'],
    columns='mask_policy',
    values=['eligible_count', 'mae_mph', 'rmse_mph'],
).round(6)
comparison

eligible_count  \
mask_policy                all_finite_targets_including_zeros   
model              horizon                                      
historical_average +15                              1416708.0   
                   +30                              1416708.0   
                   +60                              1416708.0   
persistence        +15                              1416708.0   
                   +30                              1416708.0   
                   +60                              1416708.0   

                                                   \
mask_policy                finite_nonzero_targets   
model              horizon                          
historical_average +15                  1245040.0   
                   +30                  1245022.0   
                   +60                  1244986.0   
persistence        +15                  1245040.0   
                   +30                  1245022.0   
                   +60                  1244986.0   

                                                      mae_mph  \
mask_policy                all_finite_targets_including_zeros   
model              horizon                                      
historical_average +15                              12.952750   
                   +30                              12.953424   
                   +60                              12.955054   
persistence        +15                               4.056422   
                   +30                               5.378739   
                   +60                               7.425362   

                                                   \
mask_policy                finite_nonzero_targets   
model              horizon                          
historical_average +15                   7.110352   
                   +30                   7.110392   
                   +60                   7.110807   
persistence        +15                   4.017540   
                   +30                   5.095234   
                   +60                   6.798028   

                                                     rmse_mph  \
mask_policy                all_finite_targets_including_zeros   
model              horizon                                      
historical_average +15                              21.659289   
                   +30                              21.660309   
                   +60                              21.662378   
persistence        +15                               9.929888   
                   +30                              12.807904   
                   +60                              16.334486   

                                                   
mask_policy                finite_nonzero_targets  
model              horizon                         
historical_average +15                   9.831092  
                   +30                   9.831118  
                   +60                   9.831346  
persistence        +15                   8.695499  
                   +30                  11.135297  
                   +60                  14.213606

## 7. Interpretation guardrails

Use the displayed values as simple reference baselines, not as evidence about graph usefulness by themselves. Persistence can be strong for short horizons because traffic speeds change gradually. Historical Average represents recurring weekly patterns but cannot react to current incidents or unusual congestion. The eventual XGBoost, temporal, and ST-GNN models must be evaluated against these exact saved timestamps, targets, horizons, and mask definitions.

In [6]:
assert set(metrics['model']) == {'persistence', 'historical_average'}
assert set(metrics['horizon']) == {'+15', '+30', '+60'}
assert set(metrics['mask_policy']) == {
    'finite_nonzero_targets',
    'all_finite_targets_including_zeros',
}
assert len(metrics) == 12
assert (metrics[['mae_mph', 'rmse_mph']] >= 0).all().all()

print('Task 04 notebook checks passed.')
print('Review the tables above before using these artifacts in later model comparisons.')

Task 04 notebook checks passed.
Review the tables above before using these artifacts in later model comparisons.
